# Current COMPASS abstract numbers

This notebook reads the current, already-generated COMPASS artifacts and prints the quantities used in the submitted abstract. It does **not** refit a model.

The enrichment analysis reproduces **Figure 2 v3**: classifier-derived labels from `LLM_NEPC_classifier_labels.tsv`, the full ICD-C61 `ADT_EXPOSED == 1` universe (without the prediction-cohort restriction), the figure pipeline's biomarker-label normalization, and current `platinum_MRN_list.csv` membership. Model and association results report both independent ADT-arm endpoint trees at day 0 and day 90.


In [ ]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from IPython.display import display

# Override this one path if the COMPASS output tree is mounted elsewhere.
DATA_ROOT = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS")
ENDPOINTS = ("platinum", "nepc")
RESULTS_ROOTS = {
    "platinum": DATA_ROOT / "survival_analysis/local_runs_adt",
    "nepc": DATA_ROOT / "survival_analysis/local_runs_adt_nepc",
}
LLM_ANNOTATIONS_ROOT = Path("/data/gusev/USERS/jpconnor/data/LLM_annotations/LLM_NEPC_labels")

PATHS = {
    "labels": LLM_ANNOTATIONS_ROOT / "LLM_NEPC_classifier_labels.tsv",
    "platinum": DATA_ROOT / "mrn_lists/platinum_MRN_list.csv",
    "flags": DATA_ROOT / "mrn_lists/icd_prostate_mrn_flags.csv",
}

ID_COL = "DFCI_MRN"
LANDMARKS = (0, 90)
CORE_LABELS = ("conventional", "avpc", "nepc")

missing = {name: path for name, path in PATHS.items() if not path.exists()}
if missing:
    details = "\n".join(f"  {name}: {path}" for name, path in missing.items())
    raise FileNotFoundError(
        "Required COMPASS inputs are not mounted. Set DATA_ROOT above. Missing:\n" + details
    )

print(f"Reading current artifacts beneath {DATA_ROOT}")
for name, path in PATHS.items():
    stamp = datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds")
    print(f"  {name:10s} {stamp}  {path}")


## Cohort and platinum enrichment

Percentages below use all Figure 2 v3 patients with one of the four classified primary labels in each platinum group as the denominator, including `biomarker` rows. `NEPC/AVPC` is the sum of the mutually exclusive `nepc` and `avpc` primary labels. The table also prints labeled coverage against the complete ADT-exposed universe.


In [ ]:
def normalize_id(series):
    # String normalization avoids float-vs-integer merge failures without exposing MRNs.
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype("string")

labels = pd.read_csv(PATHS["labels"], sep="\t")
flags = pd.read_csv(PATHS["flags"], usecols=[ID_COL, "ADT_EXPOSED"])
platinum = pd.read_csv(PATHS["platinum"], usecols=[ID_COL])

for frame in (labels, flags, platinum):
    frame[ID_COL] = normalize_id(frame[ID_COL])

def clean_unique_ids(frame, source_name):
    # Missing IDs cannot enter an MRN-based cohort; exact repeated records carry no new data.
    # Conflicting records remain a hard error because choosing one would change the results.
    missing_count = int(frame[ID_COL].isna().sum())
    if missing_count:
        print(f"{source_name}: dropping {missing_count:,} row(s) with a missing or non-numeric {ID_COL}")
        frame = frame.loc[frame[ID_COL].notna()].copy()

    before = len(frame)
    frame = frame.drop_duplicates().copy()
    exact_duplicate_count = before - len(frame)
    if exact_duplicate_count:
        print(f"{source_name}: collapsing {exact_duplicate_count:,} exact duplicate row(s)")

    conflicting = frame[ID_COL].duplicated(keep=False)
    if conflicting.any():
        conflicting_ids = int(frame.loc[conflicting, ID_COL].nunique())
        conflicting_rows = int(conflicting.sum())
        raise ValueError(
            f"{source_name} contains {conflicting_rows:,} non-identical rows for "
            f"{conflicting_ids:,} MRN(s). Resolve those conflicting source records; "
            "the notebook will not choose a patient label arbitrarily."
        )
    return frame

labels = clean_unique_ids(labels, PATHS["labels"].name)
flags = clean_unique_ids(flags, PATHS["flags"].name)

required = {ID_COL, "primary_label", "has_nepc", "has_avpc"}
if missing_columns := required - set(labels.columns):
    raise ValueError(f"Classifier labels are missing columns: {sorted(missing_columns)}")

# Exact Python equivalent of normalize_classifier_primary_labels() used by Figure 2 v3.
biomarker_candidates = ["biomarker_genes", "reported_biomarkers", "biomarkers_reported",
                         "reported_biomarker", "biomarkers", "biomarker"]
columns_by_lower = {column.lower(): column for column in labels.columns}
biomarker_col = next((columns_by_lower[name] for name in biomarker_candidates if name in columns_by_lower), None)
labels["primary_label"] = labels["primary_label"].astype("string").str.strip().str.lower()
labels.loc[labels["primary_label"].isin(["", "nan", "na", "null", "none"]), "primary_label"] = pd.NA
labels = labels.loc[labels["primary_label"].notna()].copy()  # load_llm_strata() drops these rows
if labels["primary_label"].eq("biomarker").any() and biomarker_col is None:
    raise ValueError("A biomarker primary label is present but no reported-biomarker column was found")
labels["has_nepc"] = pd.to_numeric(labels["has_nepc"], errors="coerce")
labels["has_avpc"] = pd.to_numeric(labels["has_avpc"], errors="coerce")
biomarker_text = labels[biomarker_col].fillna("").astype(str).str.upper() if biomarker_col else pd.Series("", index=labels.index)
qualifying = biomarker_text.str.contains(r"(?:^|[^A-Z0-9])(?:BRCA1|BRCA2|PTEN|TP53|RB1)(?:[^A-Z0-9]|$)", regex=True)
fallback = labels["primary_label"].eq("biomarker") & ~qualifying
labels.loc[fallback & labels["has_nepc"].eq(1), "primary_label"] = "nepc"
labels.loc[fallback & labels["primary_label"].eq("biomarker") & labels["has_avpc"].eq(1), "primary_label"] = "avpc"
labels.loc[fallback & labels["primary_label"].eq("biomarker"), "primary_label"] = "conventional"

adt_ids = set(flags.loc[pd.to_numeric(flags["ADT_EXPOSED"], errors="coerce").eq(1), ID_COL].dropna())
v3_labels_all = labels.loc[labels[ID_COL].isin(adt_ids)].copy()
platinum_ids = set(platinum[ID_COL].dropna())
v3_labels_all["platinum"] = v3_labels_all[ID_COL].isin(platinum_ids)
cohort = v3_labels_all.loc[v3_labels_all["primary_label"].isin(["conventional", "avpc", "nepc", "biomarker"])].copy()

print(f"Figure 2 v3 ADT-exposed universe: {len(adt_ids):,}")
print(f"Classifier rows in that universe: {len(v3_labels_all):,}")
print(f"Rows with a modeled primary-label class: {len(cohort):,}")

counts = (
    cohort.groupby(["platinum", "primary_label"], observed=False)
    .size().unstack(fill_value=0)
    .reindex(index=[True, False], fill_value=0)
    .reindex(columns=["conventional", "avpc", "nepc", "biomarker"], fill_value=0)
)
counts["NEPC/AVPC"] = counts["nepc"] + counts["avpc"]
counts["total"] = counts[["conventional", "avpc", "nepc", "biomarker"]].sum(axis=1)
percentages = 100 * counts.div(counts["total"], axis=0)

enrichment_table = pd.concat(
    {"n": counts, "percent": percentages}, axis=1
).loc[:, pd.IndexSlice[:, ["total", "NEPC/AVPC", "nepc", "avpc", "conventional", "biomarker"]]]
enrichment_table.index = ["platinum-treated", "platinum-naive"]
display(enrichment_table.round(1))

# Supplementary Figure 2 v3 binary annotations. Unlike primary_label, these can overlap.
binary_rows = []
for is_platinum, group_name in [(True, "platinum-treated"), (False, "platinum-naive")]:
    group = v3_labels_all.loc[v3_labels_all["platinum"].eq(is_platinum)]
    for annotation in ["has_nepc", "has_avpc"]:
        evaluable = pd.to_numeric(group[annotation], errors="coerce")
        evaluable = evaluable.loc[evaluable.isin([0, 1])]
        binary_rows.append({
            "platinum_group": group_name, "annotation": annotation,
            "positive_n": int(evaluable.eq(1).sum()), "evaluable_n": len(evaluable),
            "positive_percent": 100 * evaluable.eq(1).mean() if len(evaluable) else np.nan,
        })
print("Figure 2 v3 binary annotation rates (supplementary; calls may overlap):")
display(pd.DataFrame(binary_rows).round(1))


In [ ]:
n_total = len(cohort)
n_platinum = int(cohort["platinum"].sum())
pct_platinum = 100 * n_platinum / n_total

def group_values(is_platinum):
    row = percentages.loc[is_platinum]
    return row["NEPC/AVPC"], row["nepc"], row["avpc"]

treated = group_values(True)
naive = group_values(False)
cohort_sentence = (
    f"Among {n_total:,} classified patients in the Figure 2 v3 ADT-exposed prostate-cancer universe, "
    f"{n_platinum:,} ({pct_platinum:.1f}%) initiated platinum. NEPC/AVPC was identified "
    f"in {treated[0]:.1f}% of platinum-treated patients (NEPC {treated[1]:.1f}%, "
    f"AVPC {treated[2]:.1f}%) versus {naive[0]:.1f}% of platinum-naive patients "
    f"(NEPC {naive[1]:.1f}%, AVPC {naive[2]:.1f}%)."
)
print(cohort_sentence)


## Held-out model performance

The `both` configuration is the lab-feature model used for the headline comparison; `baseline` is shown as a useful age-only check. The table reports the held-out test metrics, never cross-validation performance.


In [ ]:
MODEL_FILES = {
    ("elastic-net", "both"): ("cox", "cox_agg_multivariable_metrics.csv"),
    ("elastic-net", "baseline"): ("cox", "cox_agg_baseline_metrics.csv"),
    ("xgboost", "both"): ("xgboost", "landmark_xgboost_metrics.csv"),
    ("xgboost", "baseline"): ("xgboost", "landmark_xgboost_baseline_metrics.csv"),
}

# Every model family writes one canonical metrics schema
# (survival_common/metrics_schema.py), so there are no per-family spellings to
# fall back through. Metrics CSVs written before that cutover lack these
# columns and read as NaN; refit rather than re-adding fallbacks.
def scalar(row, name):
    if name in row.index and pd.notna(row[name]):
        return float(row[name])
    return np.nan

model_rows = []
for endpoint, results_root in RESULTS_ROOTS.items():
  for landmark in LANDMARKS:
    for (model, config), (subdir, filename) in MODEL_FILES.items():
        path = results_root / subdir / f"landmark_{landmark}" / config / filename
        base = dict(endpoint=endpoint, landmark_days=landmark, model=model, configuration=config, source=str(path))
        if not path.exists():
            model_rows.append({**base, "status": "missing"})
            continue
        frame = pd.read_csv(path)
        if "endpoint" not in frame or not frame["endpoint"].astype(str).str.lower().eq(endpoint).any():
            model_rows.append({**base, "status": f"no {endpoint} row"})
            continue
        row = frame.loc[frame["endpoint"].astype(str).str.lower().eq(endpoint)].iloc[0]
        model_rows.append({
            **base,
            "n_test": scalar(row, "n_test"),
            "n_test_events": scalar(row, "n_events_test"),
            "c_index": scalar(row, "test_c_index"),
            "mean_auc_t": scalar(row, "test_mean_auc_t"),
            "integrated_brier": scalar(row, "test_integrated_brier"),
            "status": "ok",
        })

model_results = pd.DataFrame(model_rows)
display(model_results.drop(columns="source").round(3))

headline_xgb = model_results.query(
    "landmark_days == 0 and model == 'xgboost' and configuration == 'both' and status == 'ok'"
)
if headline_xgb.empty:
    print("Headline XGBoost day-0 result is unavailable (see status table).")
else:
    xgb_auc = headline_xgb.iloc[0]["mean_auc_t"]
    display(headline_xgb[["endpoint", "mean_auc_t", "c_index", "integrated_brier"]].round(3))


## Testosterone and PSA associations

These are the current full-cohort, age- and observation-count-adjusted univariate Cox estimates. An HR above 1 means a one-SD increase in the named raw feature is associated with faster platinum initiation; an HR below 1 means it is associated with slower platinum initiation. For `delta`, the raw feature is `last − first`.


In [ ]:
TARGET_ASSOCIATIONS = pd.DataFrame([
    {"landmark_days": 0,  "lab_name": "Testosterone", "feature_stat": "mean"},
    {"landmark_days": 90, "lab_name": "Testosterone", "feature_stat": "mean"},
    {"landmark_days": 90, "lab_name": "PSA",          "feature_stat": "last"},
    {"landmark_days": 90, "lab_name": "PSA",          "feature_stat": "delta"},
])

association_frames = []
for endpoint, results_root in RESULTS_ROOTS.items():
  for landmark in LANDMARKS:
    path = results_root / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_univariate_nobs_adjusted.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing current univariate results: {path}")
    frame = pd.read_csv(path)
    frame = frame.loc[frame["endpoint"].astype(str).str.lower().eq(endpoint)].copy()
    frame["endpoint"] = endpoint
    frame["landmark_days"] = landmark
    frame["source"] = str(path)
    association_frames.append(frame)

associations = pd.concat(association_frames, ignore_index=True)
associations["lab_key"] = associations["lab_name"].astype(str).str.casefold()
associations["stat_key"] = associations["feature_stat"].astype(str).str.casefold()
targets = TARGET_ASSOCIATIONS.merge(pd.DataFrame({"endpoint": ENDPOINTS}), how="cross").assign(
    lab_key=lambda d: d["lab_name"].str.casefold(),
    stat_key=lambda d: d["feature_stat"].str.casefold(),
)
selected = targets.merge(
    associations, on=["endpoint", "landmark_days", "lab_key", "stat_key"],
    how="left", validate="one_to_one", suffixes=("_requested", "")
)
missing_rows = selected["hazard_ratio_per_sd"].isna()
if missing_rows.any():
    requested = selected.loc[missing_rows, ["endpoint", "landmark_days", "lab_name_requested", "feature_stat_requested"]]
    raise KeyError("Requested abstract association rows were not found:\n" + requested.to_string(index=False))

association_table = selected[[
    "endpoint", "landmark_days", "lab_name_requested", "feature_stat_requested", "feature",
    "n_patients_used", "n_events_used", "hazard_ratio_per_sd",
    "ci_lower", "ci_upper", "p_value", "q_value", "note", "source"
]].rename(columns={"lab_name_requested": "lab_name", "feature_stat_requested": "feature_stat"})
display(association_table.drop(columns="source"))


In [ ]:
def scientific(value, digits=1):
    if pd.isna(value):
        return "NA"
    return f"{value:.{digits}e}"

def direction_phrase(row):
    hr = float(row["hazard_ratio_per_sd"])
    lab = row["lab_name"]
    stat = row["feature_stat"]
    if stat == "delta":
        subject = f"increases in {lab}" if hr > 1 else f"decreases in {lab}"
    elif stat == "last":
        subject = f"higher last {lab}" if hr > 1 else f"lower last {lab}"
    else:
        subject = f"higher mean {lab}" if hr > 1 else f"lower mean {lab}"
    # Express the HR for the direction named in the prose. If the raw-feature HR is
    # below 1, the reciprocal describes the corresponding lower/decrease contrast.
    prose_hr = hr if hr > 1 else 1 / hr
    return subject, prose_hr

phrases = []
for _, row in association_table.iterrows():
    subject, prose_hr = direction_phrase(row)
    when = "baseline" if row["landmark_days"] == 0 else "3 months"
    phrases.append(
        f"{row['endpoint']} / {when}: {subject} was associated with faster time-to-{row['endpoint']} "
        f"(HR {prose_hr:.2f} per SD in the named direction; raw-feature HR "
        f"{row['hazard_ratio_per_sd']:.2f}, q={scientific(row['q_value'])})."
    )

print("\n".join(phrases))


## Draft updated results paragraph

This final cell assembles the current values. It retains explicit raw-feature HRs whenever the prose names the inverse contrast (for example, “lower PSA”), avoiding ambiguity about which variable the Cox model actually fit.


In [ ]:
model_sentence = ""
if not headline_xgb.empty:
    model_sentence = " " + " ".join(
        f"For {row.endpoint}, XGBoost achieved a held-out mean AUC(t) of {row.mean_auc_t:.2f}."
        for row in headline_xgb.itertuples()
    )

association_sentence = " ".join(phrases)
updated_paragraph = cohort_sentence + model_sentence + " " + association_sentence
print(updated_paragraph)

print("\nSource files used:")
used_sources = list(PATHS.values()) + [Path(p) for p in model_results.loc[model_results.status.eq("ok"), "source"]] + [Path(p) for p in association_table["source"]]
for path in dict.fromkeys(used_sources):
    print(f"  {path}")
